In [2]:
import torch

if torch.cuda.is_available():
    print(f"CUDA is available. You have {torch.cuda.device_count()} GPU(s).")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available. You are using the CPU.")

CUDA is available. You have 1 GPU(s).
GPU Name: NVIDIA GeForce RTX 3080


In [15]:
import napari
from napari.utils.colormaps import Colormap, DirectLabelColormap
from qtpy.QtWidgets import QWidget, QVBoxLayout, QHBoxLayout, QPushButton, QLabel, QStackedWidget, QMessageBox, QLineEdit, QComboBox, QCheckBox
from magicgui.widgets import ComboBox, FileEdit
import os
import glob
import numpy as np
import tifffile
from skimage.filters import threshold_otsu, threshold_triangle, threshold_mean, threshold_minimum, threshold_li, threshold_yen
import torch
from segmentation import cellpose_live_segmentation
from tracking import generate_trackmate_labels



In [ ]:
class MyTool(QWidget):
    def __init__(self, viewer):
        super().__init__()
        self.viewer = viewer
        self.state = {}
        self.current_image = None
        # Stack of pages visited, so Back always returns to where the user came from.
        self.history = []

        # Holds all pages
        self.pages = QStackedWidget()
        self.upload_page = self.create_upload_page()
        self.pages.addWidget(self.upload_page)

        layout = QVBoxLayout()
        layout.addWidget(self.pages)
        self.setLayout(layout)

    def go_back(self):
        if self.history:
            self.pages.setCurrentIndex(self.pages.indexOf(self.history.pop()))

    # PAGE 1 - Upload Data: Either Image or Stack
    # self.upload_type = "Image" or "Folder"; self.inputs maps name -> input widgets
    # (self.inputs[name]["image"] / ["folder"]). self.label_status / self.tracks_status
    # flag whether inputs are pre-labelled / pre-tracked.
    def change_upload_type(self):
        is_image = self.upload_type.value == "Image"
        
        for widgets in self.inputs.values():
            widgets["image"].visible = is_image
            widgets["folder"].visible = not is_image
            if is_image:
                widgets["label"].setText(widgets["image_label"])
            else:
                widgets["label"].setText(widgets["folder_label"])


    def create_upload_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.addWidget(QLabel("Upload Your Data"))

        # Image / Folder selector
        self.upload_type = ComboBox(label="Input type:", choices=["Image", "Folder"], value="Image")
        layout.addWidget(self.upload_type.native)

        # Store our input widgets
        self.inputs = {}

        # Add the three rows
        self.add_upload_row(layout, "original", "Original Image (Stack)", "(Folder of) Original Images")
        self.add_upload_row(layout, "labels", "(Stack of) Labels", "(Folder of) Labels")
        self.add_upload_row(layout, "tracks", "(Stack of) Linked Tracks", "(Folder of) Linked Tracks")

        # Update everything when Image/Folder changes
        self.upload_type.changed.connect(self.change_upload_type)

        # Next button
        next_button = QPushButton("Next →")
        next_button.clicked.connect(self.check_upload_page)
        layout.addWidget(next_button)

        return page

    def add_upload_row(self, layout, name, image_label_text, folder_label_text):
        row = QHBoxLayout()
        label = QLabel(image_label_text)
        image_input = FileEdit(label="", mode="r", filter="TIFF (*.tif *.tiff)")
        folder_input = FileEdit(label="", mode="d")

        # Default to image input, so hide the folder input intiailly 
        folder_input.visible = False

        row.addWidget(label)
        row.addWidget(image_input.native)
        row.addWidget(folder_input.native)

        layout.addLayout(row)

        self.inputs[name] = {
            "label": label,
            "image": image_input,
            "folder": folder_input,
            "image_label": image_label_text,
            "folder_label": folder_label_text
        }
        
    def is_empty(self, value):
        # magicgui FileEdit returns a Path; an unset field is Path('.'), not "".
        return str(value).strip() in ("", ".")

    def tyx_size(self, tif_path):
        # (T, Y, X) sizes only; channel axis ignored so (T,C,Y,X) matches (T,1,Y,X).
        with tifffile.TiffFile(tif_path) as tif:
            series = tif.series[0]
            if "S" in series.axes:
                t_dim = series.axes.index("S")
            elif "T" in series.axes:
                t_dim = series.axes.index("T")
            else:
                t_dim = 1
            sizes = dict(zip(series.axes, series.shape))
        return (t_dim, sizes.get("Y"), sizes.get("X"))

    def channel_count(self, tif_path):
        # Number of channels (C axis); images without a C axis count as 1.
        with tifffile.TiffFile(tif_path) as tif:
            series = tif.series[0]
            sizes = dict(zip(series.axes, series.shape))
        return sizes.get("C", 1)

    def to_tcyx(self, tif_path):
        # Return array as (T, C, Y, X); missing T/C axes are inserted as size 1.
        with tifffile.TiffFile(tif_path) as tif:
            series = tif.series[0]
            image_array = series.asarray()
            raw_axes = series.axes
            dims = list(raw_axes)
        # tifffile may label a stray frame/sample axis 'S'/'I'/'Q'; treat it as time.
        dims = ["T" if a in ("S", "I", "Q") else a for a in dims]
        # Drop any remaining non-standard singleton axes (e.g. a size-1 Z).
        for axis, size in list(zip(list(dims), image_array.shape)):
            if axis not in ("T", "C", "Y", "X") and size == 1:
                index = dims.index(axis)
                image_array = np.squeeze(image_array, axis=index)
                dims.pop(index)
        if "Y" not in dims or "X" not in dims or set(dims) - {"T", "C", "Y", "X"}:
            raise ValueError(f"Unsupported TIFF axes '{raw_axes}' for {tif_path}.")
        for axis in ("C", "T"):
            if axis not in dims:
                image_array = np.expand_dims(image_array, 0)
                dims.insert(0, axis)
        order = [dims.index(a) for a in ("T", "C", "Y", "X")]
        return np.transpose(image_array, order)

    def check_upload_page(self):
        try:
            ###################TO DO ------ CAN'T JUST HAVE A SINGLE CHANNEL IMAGE ############
            ############ IF T && Z -> FAIL
            ### FAILURE MODES - no inputs
            if (self.upload_type.value == "Image" and self.is_empty(self.inputs["original"]["image"].value)) or (self.upload_type.value == "Folder" and self.is_empty(self.inputs["original"]["folder"].value)):
                QMessageBox.warning(self, "Input Error", "Please select an original image/folder.")
                return
            ########## FOLDER FAILURE MODES: empty folders, mismatched length in folders, same folder listed multiple times
            if self.upload_type.value == "Folder":
                folders = {
                    "Original": self.inputs["original"]["folder"].value,
                    "Labels": self.inputs["labels"]["folder"].value,
                    "Tracks": self.inputs["tracks"]["folder"].value,
                }
                # Unset FileEdit is Path('.'); normalise blanks to "" so they aren't treated as a real (duplicate) folder.
                folders = {name: ("" if self.is_empty(value) else str(value)) for name, value in folders.items()}
                # Check for duplicate folders
                folder_paths = [folder for folder in folders.values() if folder]
                if len(folder_paths) != len(set(folder_paths)):
                    QMessageBox.warning(self, "Input Error", "The same folder has been listed multiple times.")
                    return

                n_original = len(glob.glob(os.path.join(folders["Original"], "*.tif"))) + len(glob.glob(os.path.join(folders["Original"], "*.tiff")))
                n_labels = len(glob.glob(os.path.join(folders["Labels"], "*.tif"))) + len(glob.glob(os.path.join(folders["Labels"], "*.tiff"))) if folders["Labels"] else 0
                n_tracks = len(glob.glob(os.path.join(folders["Tracks"], "*.tif"))) + len(glob.glob(os.path.join(folders["Tracks"], "*.tiff"))) if folders["Tracks"] else 0
                folder_lengths = [n_original]
                if n_labels:
                    folder_lengths.append(n_labels)
                if n_tracks:
                    folder_lengths.append(n_tracks)
                if set(folder_lengths) == {0}:
                    QMessageBox.warning(self, "Input Error", "All selected folders are empty.")
                    return
                if len(set(folder_lengths)) > 1:
                    QMessageBox.warning(self, "Input Error", "Folders contain different numbers of images.")
                    return

                self.original_images = glob.glob(os.path.join(folders["Original"], "*.tif")) + glob.glob(os.path.join(folders["Original"], "*.tiff"))
                self.label_images = glob.glob(os.path.join(folders["Labels"], "*.tif")) + glob.glob(os.path.join(folders["Labels"], "*.tiff")) if folders["Labels"] else []
                self.track_images = glob.glob(os.path.join(folders["Tracks"], "*.tif")) + glob.glob(os.path.join(folders["Tracks"], "*.tiff")) if folders["Tracks"] else []

                # Every original image in the folder must have the same number of channels.
                channel_counts = {self.channel_count(image) for image in self.original_images}
                if len(channel_counts) > 1:
                    QMessageBox.warning(self, "Input Error", "Original images have different numbers of channels.")
                    return
            else:
                ######### IMAGE FAILURE MODES - same path repeated
                self.original_images = [] if self.is_empty(self.inputs["original"]["image"].value) else [self.inputs["original"]["image"].value]
                self.label_images = [] if self.is_empty(self.inputs["labels"]["image"].value) else [self.inputs["labels"]["image"].value]
                self.track_images = [] if self.is_empty(self.inputs["tracks"]["image"].value) else [self.inputs["tracks"]["image"].value]

                paths = [str(p) for p in (self.inputs["original"]["image"].value, self.inputs["labels"]["image"].value, self.inputs["tracks"]["image"].value) if not self.is_empty(p)]
                if len(paths) != len(set(paths)):
                    QMessageBox.warning(self, "Input Error", "You've listed the same file in multiple categories! Ensure paths are unique.")
                    return

            if (self.upload_type.value == "Image" and self.is_empty(self.inputs["labels"]["image"].value)) or (self.upload_type.value == "Folder" and self.is_empty(self.inputs["labels"]["folder"].value)):
                self.label_status = False
            else:
                self.label_status = True
                self.label_image = self.to_tcyx(self.label_images[0])
            if (self.upload_type.value == "Image" and self.is_empty(self.inputs["tracks"]["image"].value)) or (self.upload_type.value == "Folder" and self.is_empty(self.inputs["tracks"]["folder"].value)):
                self.tracks_status = False
            else:
                self.tracks_status = True
                self.track_image = self.to_tcyx(self.track_images[0])

            self.original_images.sort()
            self.label_images.sort()
            self.track_images.sort()
            for i, image in enumerate(self.original_images):
                original_shape = self.tyx_size(image)
                label_shape = self.tyx_size(self.label_images[i]) if self.label_status else None
                track_shape = self.tyx_size(self.track_images[i]) if self.tracks_status else None
                shapes = [original_shape]
                if label_shape is not None:
                    shapes.append(label_shape)
                if track_shape is not None:
                    shapes.append(track_shape)
                if len(set(shapes)) > 1:
                    QMessageBox.warning(self, "Input Error", f"(T, Y, X) sizes do not match for {image}:\n"
                        f"Original: {original_shape}\n"
                        f"Label: {label_shape if label_shape is not None else 'N/A'}\n"
                        f"Track: {track_shape if track_shape is not None else 'N/A'}"
                    )
                    return

            if getattr(self, "threshold_page", None) is not None:
                self.pages.removeWidget(self.threshold_page)
                self.threshold_page.deleteLater()
            self.threshold_page = self.create_threshold_page()
            self.pages.addWidget(self.threshold_page)
            self.history.append(self.pages.currentWidget())
            self.pages.setCurrentIndex(self.pages.indexOf(self.threshold_page))
        except Exception as exception:
            import traceback
            traceback.print_exc()
            QMessageBox.warning(self, "Error", f"{type(exception).__name__}: {exception}")


    ####### PAGE 2 - Threshold and Channel Assignment
    # --------------------------
    def map_channels_to_plotting_colours(self, channel_name):
        channel_name = channel_name.lower()
        colour_map = {
            "dodgerblue": ["dapi", "hoechst", "405", "blue"],
            "limegreen":  ["fitc", "gfp", "488", "green"],
            "red":        ["tritc", "rfp", "561", "555", "mcherry", "mscarlet", "tomato", "red"],
            "deeppink":   ["cy5", "647", "640", "far red", "far-red", "magenta"],
            "darkorange": ["cy3", "orange", "561", "568", "yellow", "yfp"],
            "gray":       ["bf", "brightfield", "phase"]
        }
        for colour, keywords in colour_map.items():
            if any(keyword in channel_name for keyword in keywords):
                return colour
        return "gray"

    def map_threshold_to_method(self, threshold):
        threshold_map = {
            "otsu": threshold_otsu,
            "mean":  threshold_mean,
            "yen":   threshold_yen,
            "triangle":   threshold_triangle,
            "minimum": threshold_minimum,
            "numeric": lambda image, value: (image > float(value))
        }
        return threshold_map.get(threshold.lower())

    
    def add_threshold_row(self, layout,  default_channel=0, default_name="", default_threshold="Otsu"):
        row = QHBoxLayout()
        n_channels = self.current_image.shape[1]
        # Channel dropdown: 0 ... n_channels-1
        channel_input = QComboBox()
        channel_input.addItems([str(i) for i in range(n_channels)])
        channel_input.setCurrentIndex(default_channel)

        # Channel name
        name_input = QLineEdit()
        name_input.setText(default_name)
        name_input.setPlaceholderText("Channel name")

        # Threshold method
        threshold_input = QComboBox()
        threshold_input.addItems(["Mean", "Otsu", "Yen", "Triangle", "Minimum", "Numeric"])
        threshold_input.setCurrentText(default_threshold)


        ######If they select "Numeric", we need another box to appear that allows them to enter a number
        numeric_input = QLineEdit()
        numeric_input.setPlaceholderText("Enter numeric threshold")
        numeric_input.setVisible(False)

        def on_threshold_change(index):
            if threshold_input.currentText() == "Numeric":
                numeric_input.setVisible(True)
            else:
                numeric_input.setVisible(False)

        threshold_input.currentIndexChanged.connect(on_threshold_change)

        row.addWidget(channel_input)
        row.addWidget(name_input)
        row.addWidget(threshold_input)
        row.addWidget(numeric_input)

        layout.addLayout(row)
        self.threshold_rows.append({
            "channel": channel_input,
            "name": name_input,
            "threshold": threshold_input,
            "numeric": numeric_input,
        })



    def check_and_apply_thresholds(self):
        try:
            self.channel_settings = {}
            used_channels = []
            used_colours = []

            for row in self.threshold_rows:
                # Get values from GUI
                channel = int(row["channel"].currentText())
                name = row["name"].text().strip()
                threshold = row["threshold"].currentText()
                # Blanks are OK: skip any row the user left unnamed.
                if not name:
                    continue
                # Channel must not be used twice
                if channel in used_channels:
                    QMessageBox.warning(self, "Invalid Channels", f"Channel {channel} has been selected more than once.")
                    return False
                used_channels.append(channel)

                # Numeric threshold requires a number
                if threshold == "Numeric":
                    try:
                        threshold_value = float(row["numeric"].text())
                    except ValueError:
                        QMessageBox.warning(self, "Invalid Threshold", f"Please enter a numeric threshold for channel {channel}.")
                        return False
                else:
                    threshold_value = threshold

                colour = self.map_channels_to_plotting_colours(name)
                # Keep every channel a different colour: if this one is taken, grab an unused one.
                if colour in used_colours:
                    for candidate in ["dodgerblue", "limegreen", "red", "deeppink", "darkorange", "gray"]:
                        if candidate not in used_colours:
                            colour = candidate
                            break
                used_colours.append(colour)

                self.channel_settings[channel] = {
                    "name": name,
                    "colour": colour,
                    "threshold": threshold_value,
                }

            # Show the original channels (named from the entry boxes) plus a mask per named fluorescent channel.
            self.viewer.layers.clear()
            bf_channel = int(self.bf_channel_input.currentText())
            bf_name = self.bf_name_input.text().strip()
            names = []
            colormaps = []
            for channel in range(self.current_image.shape[1]):
                if channel in self.channel_settings:
                    names.append(self.channel_settings[channel]["name"])
                    colour = self.channel_settings[channel]["colour"]
                elif channel == bf_channel and bf_name:
                    names.append(bf_name)
                    colour = "gray"
                else:
                    names.append(f"Channel {channel}")
                    colour = "gray"
                # Black-to-colour ramp so each channel is displayed in its label colour.
                colormaps.append(Colormap(["black", colour], name=f"channel_{channel}"))
            self.viewer.add_image(self.current_image, channel_axis=1, name=names, colormap=colormaps)

            for channel, settings in self.channel_settings.items():
                channel_image = self.current_image[:, channel]
                threshold = settings["threshold"]
                method = self.map_threshold_to_method(threshold if isinstance(threshold, str) else "numeric")
                if isinstance(threshold, str):
                    mask = channel_image > method(channel_image)
                else:
                    mask = method(channel_image, threshold)
                colormap = DirectLabelColormap(color_dict={1: settings["colour"], None: "transparent"})
                self.viewer.add_labels(mask.astype("uint8"), name=f"{settings['name']}_segmentation", colormap=colormap)

            return True
        except Exception as exception:
            import traceback
            traceback.print_exc()
            QMessageBox.warning(self, "Error", f"{type(exception).__name__}: {exception}")
            return False

    def create_threshold_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)

        layout.addWidget(QLabel("Channels and Thresholds"))

        try:
            self.current_image = self.to_tcyx(self.original_images[0])
            n_channels = self.current_image.shape[1]

            # Brightfield / segmentation channel (# + name): only needed when the user hasn't supplied labels.
            if not self.label_status:
                bf_label = QLabel("Channel To Segment Cells/Nuclei From")
            else:
                bf_label = QLabel("Channel Your Labels were Built From (Visualisation Only)") 
            layout.addWidget(bf_label)
            bf_row = QHBoxLayout()
            self.bf_channel_input = QComboBox()
            self.bf_channel_input.addItems([str(i) for i in range(n_channels)])
            self.bf_channel_input.setCurrentIndex(0)
            self.bf_name_input = QLineEdit()
            self.bf_name_input.setPlaceholderText("Brightfield")
            bf_row.addWidget(self.bf_channel_input)
            bf_row.addWidget(self.bf_name_input)
            layout.addLayout(bf_row)

            layout.addWidget(QLabel("Fluorescent Channels to Analyse"))

            self.threshold_rows = [ ]
            for channel_index in range(n_channels - 1):
                self.add_threshold_row(layout, default_channel=channel_index + 1, default_name=f"Channel {channel_index + 1}")

            ### BUTTON FOR APPLYING THRESHOLDS
            apply_thresholds_button = QPushButton("Apply Thresholds")
            layout.addWidget(apply_thresholds_button)

            # Next only becomes available once thresholds have been applied and validated.
            next_button = QPushButton("Next →")
            next_button.setEnabled(False)
            layout.addWidget(next_button)

            def apply_thresholds():
                next_button.setEnabled(self.check_and_apply_thresholds())

            apply_thresholds_button.clicked.connect(apply_thresholds)
            # Skip cellpose when labels exist; skip cellpose + trackmate when tracks exist.
            if self.label_status == False and self.tracks_status == False:
                next_button.clicked.connect(self.show_cellpose_page)
            elif self.label_status == True and self.tracks_status == False:
                next_button.clicked.connect(self.show_trackmate_page)
            else:
                next_button.clicked.connect(self.show_analysis_page)

        except Exception as exception:
            import traceback
            traceback.print_exc()
            QMessageBox.warning(self, "Error", f"{type(exception).__name__}: {exception}")

        back = QPushButton("← Back")
        back.clicked.connect(self.go_back)
        layout.addWidget(back)

        return page

    # --------------------------
    # Cellpose
    # --------------------------

    def show_cellpose_page(self):
        # We can navigate forwards and backwards, so delete any old instance and recreate it.
        self.history.append(self.pages.currentWidget())
        if getattr(self, "cellpose_page", None) is not None:
            self.pages.removeWidget(self.cellpose_page)
            self.cellpose_page.deleteLater()
        self.cellpose_page = self.create_cellpose_page()
        self.pages.addWidget(self.cellpose_page)
        self.pages.setCurrentIndex(self.pages.indexOf(self.cellpose_page))

    def create_cellpose_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)

        try:
            layout.addWidget(QLabel("Cellpose Segmentation"))

            # Optional custom model: the file picker stays hidden until the box is ticked.
            self.custom_model_checkbox = QCheckBox("Use custom model?")
            layout.addWidget(self.custom_model_checkbox)
            self.custom_model_input = FileEdit(label="Custom model", mode="r", filter="")
            self.custom_model_input.visible = False
            layout.addWidget(self.custom_model_input.native)
            self.custom_model_checkbox.toggled.connect(lambda checked: setattr(self.custom_model_input, "visible", checked))

            # Minimum cell size in pixels (integer only).
            size_row = QHBoxLayout()
            size_row.addWidget(QLabel("Min cell size (pixels):"))
            self.min_cell_size_input = QLineEdit()
            self.min_cell_size_input.setText("15")
            size_row.addWidget(self.min_cell_size_input)
            layout.addLayout(size_row)

            segment_button = QPushButton("Segment")
            segment_button.clicked.connect(self.run_cellpose_segmentation)
            layout.addWidget(segment_button)

            back = QPushButton("← Back")
            back.clicked.connect(self.go_back)
            layout.addWidget(back)
        except Exception as exception:
            import traceback
            traceback.print_exc()
            QMessageBox.warning(self, "Error", f"{type(exception).__name__}: {exception}")

        return page

    def run_cellpose_segmentation(self):
        try:
            # Min cell size must be a whole number of pixels.
            try:
                min_cell_size = int(self.min_cell_size_input.text())
            except ValueError:
                QMessageBox.warning(self, "Invalid Value", "Min cell size must be an integer.")
                return

            custom_model_path = None
            if self.custom_model_checkbox.isChecked():
                if self.is_empty(self.custom_model_input.value):
                    QMessageBox.warning(self, "Missing Model", "Please select a custom model file.")
                    return
                custom_model_path = str(self.custom_model_input.value)

            use_gpu = torch.cuda.is_available()
            segment_channel = int(self.bf_channel_input.currentText())

            # Save segmentations in a new folder next to the original images.
            output_folder = os.path.join(os.path.dirname(str(self.original_images[0])), "FluoroFate_Segmentation")
            os.makedirs(output_folder, exist_ok=True)

            first_masks = None
            for image_path in self.original_images:
                brightfield = self.to_tcyx(image_path)[:, segment_channel]
                masks = np.asarray(cellpose_live_segmentation(brightfield, min_size=min_cell_size, custom_model_path=custom_model_path, gpu=use_gpu))
                if masks.ndim == 2:
                    masks = masks[np.newaxis]
                name = os.path.splitext(os.path.basename(str(image_path)))[0]
                tifffile.imwrite(os.path.join(output_folder, f"{name}_segmentation.tif"), masks.astype(np.uint16))
                if first_masks is None:
                    first_masks = masks

            # Tracking reads its masks from here.
            self.label_images = sorted(glob.glob(os.path.join(output_folder, "*.tif")))
            # Overlay the labels for the single / first image.
            self.viewer.add_labels(first_masks.astype("uint32"), name="Cellpose labels")
            self.label_status = True

            self.show_trackmate_page()
        except Exception as exception:
            import traceback
            traceback.print_exc()
            QMessageBox.warning(self, "Error", f"{type(exception).__name__}: {exception}")

    # --------------------------
    # Trackmate
    # --------------------------

    def show_trackmate_page(self):
        # We can navigate forwards and backwards, so delete any old instance and recreate it.
        self.history.append(self.pages.currentWidget())
        if getattr(self, "trackmate_page", None) is not None:
            self.pages.removeWidget(self.trackmate_page)
            self.trackmate_page.deleteLater()
        self.trackmate_page = self.create_trackmate_page()
        self.pages.addWidget(self.trackmate_page)
        self.pages.setCurrentIndex(self.pages.indexOf(self.trackmate_page))

    def create_trackmate_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)

        try:
            layout.addWidget(QLabel("TrackMate Tracking"))

            radius_row = QHBoxLayout()
            radius_row.addWidget(QLabel("Initial search radius:"))
            self.initial_search_radius_input = QLineEdit()
            self.initial_search_radius_input.setText("30")
            radius_row.addWidget(self.initial_search_radius_input)
            layout.addLayout(radius_row)

            search_row = QHBoxLayout()
            search_row.addWidget(QLabel("Search radius:"))
            self.search_radius_input = QLineEdit()
            self.search_radius_input.setText("150")
            search_row.addWidget(self.search_radius_input)
            layout.addLayout(search_row)

            gap_row = QHBoxLayout()
            gap_row.addWidget(QLabel("Max frame gap:"))
            self.max_frame_gap_input = QLineEdit()
            self.max_frame_gap_input.setText("3")
            gap_row.addWidget(self.max_frame_gap_input)
            layout.addLayout(gap_row)

            self.allow_splitting_checkbox = QCheckBox("Allow track splitting?")
            self.allow_splitting_checkbox.setChecked(True)
            layout.addWidget(self.allow_splitting_checkbox)

            track_button = QPushButton("Track")
            track_button.clicked.connect(self.run_trackmate)
            layout.addWidget(track_button)

            back = QPushButton("← Back")
            back.clicked.connect(self.go_back)
            layout.addWidget(back)
        except Exception as exception:
            import traceback
            traceback.print_exc()
            QMessageBox.warning(self, "Error", f"{type(exception).__name__}: {exception}")

        return page

    def run_trackmate(self):
        try:
            # Show the uploaded labels being tracked (the Cellpose path overlays its own).
            if getattr(self, "label_image", None) is not None:
                self.viewer.add_labels(self.label_image[:, 0].astype("uint32"), name="Labels")
            # Radii are distances (floats); max frame gap is a whole number of frames.
            try:
                initial_search_radius = float(self.initial_search_radius_input.text())
                search_radius = float(self.search_radius_input.text())
                max_frame_gap = int(self.max_frame_gap_input.text())
            except ValueError:
                QMessageBox.warning(self, "Invalid Value", "Search radii must be numbers and max frame gap must be an integer.")
                return

            allow_splitting = self.allow_splitting_checkbox.isChecked()

            # Save linked tracks in a new folder next to the original images.
            output_folder = os.path.join(os.path.dirname(str(self.original_images[0])), "FluoroFate_Linked_Tracks")
            os.makedirs(output_folder, exist_ok=True)

            first_labels = None
            first_tracks = None
            imagej_instance = None
            for mask_path in self.label_images:
                name = os.path.splitext(os.path.basename(str(mask_path)))[0]
                result = generate_trackmate_labels(masks_path=mask_path, output_directory=os.path.join(output_folder, name), initial_search_radius=initial_search_radius, search_radius=search_radius, max_frame_gap=max_frame_gap, allow_track_splitting=allow_splitting, imagej_instance=imagej_instance)
                imagej_instance = result["imagej_instance"]
                if first_labels is None:
                    first_labels = result["linked_labels"]
                    first_tracks = result["trackmate_tracks_df"]

            # Overlay the linked labels + tracks for the single / first image.
            self.viewer.add_labels(np.asarray(first_labels).astype("uint32"), name="Linked labels")
            if len(first_tracks) > 0:
                tracks_array = first_tracks[["track_id", "t", "y", "x"]].to_numpy(dtype=float)
                self.viewer.add_tracks(tracks_array, name="Tracks")

            self.show_analysis_page()
        except Exception as exception:
            import traceback
            traceback.print_exc()
            QMessageBox.warning(self, "Error", f"{type(exception).__name__}: {exception}")

    # --------------------------
    # Analysis
    # --------------------------

    def show_analysis_page(self):
        # We can navigate forwards and backwards, so delete any old instance and recreate it.
        self.history.append(self.pages.currentWidget())
        if getattr(self, "analysis_page", None) is not None:
            self.pages.removeWidget(self.analysis_page)
            self.analysis_page.deleteLater()
        self.analysis_page = self.create_analysis_page()
        self.pages.addWidget(self.analysis_page)
        self.pages.setCurrentIndex(self.pages.indexOf(self.analysis_page))

    def create_analysis_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)

        try:
            layout.addWidget(QLabel("Ready to run!"))

            back = QPushButton("← Back")
            run = QPushButton("Run analysis")

            back.clicked.connect(self.go_back)
            run.clicked.connect(lambda: self.run_analysis())

            layout.addWidget(back)
            layout.addWidget(run)
        except Exception as exception:
            import traceback
            traceback.print_exc()
            QMessageBox.warning(self, "Error", f"{type(exception).__name__}: {exception}")

        return page


viewer = napari.Viewer()

tool = MyTool(viewer)

viewer.window.add_dock_widget(
    tool,
    name="FluoroFate",
    area="right",
)

napari.run()